In [ ]:
"""
LLM Agent with Tool Calling using LangGraph
============================================
Features:
- LangGraph ReAct agent in a while loop
- Document tokenisation via tiktoken
- Tools: Calculator + Notion API (real)
- Backend: Groq (FREE)

Install:
    pip install langgraph langchain langchain-groq tiktoken requests

Notion setup:
    1. https://notion.so/my-integrations -> New integration -> copy secret_...
    2. Open a Notion page -> "..." -> Connect to -> your integration
    3. Copy the page ID from the URL (32-char hex after the last dash)

Kaggle secrets to add:
    GROQ_API_KEY          = gsk_...
    NOTION_API_KEY        = secret_...
    NOTION_PARENT_PAGE_ID = xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
"""
!pip install -q langgraph langchain langchain-groq tiktoken requests
import os
import math
import tiktoken
import requests
from typing import Annotated

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from typing_extensions import TypedDict
from langchain_groq import ChatGroq

# ── Kaggle Secrets (uncomment to use) ──────────────────────────────────────
# from kaggle_secrets import UserSecretsClient
# _s = UserSecretsClient()
# os.environ["GROQ_API_KEY"]           = _s.get_secret("GROQ_API_KEY")
# os.environ["NOTION_API_KEY"]         = _s.get_secret("NOTION_API_KEY")
# os.environ["NOTION_PARENT_PAGE_ID"]  = _s.get_secret("NOTION_PARENT_PAGE_ID")

# ── Keys ───────────────────────────────────────────────────────────────────
GROQ_API_KEY          = os.getenv("GROQ_API_KEY",         "######") ##replace with groq api_key
NOTION_API_KEY        = os.getenv("NOTION_API_KEY",        "######") ##replace with notion api_key
NOTION_PARENT_PAGE_ID = os.getenv("NOTION_PARENT_PAGE_ID", "35a42318ab7e80bd8850dc9d369c2a8c")

TOKENIZER_MODEL = "gpt-4o"
MAX_AGENT_STEPS = 10
NOTION_VERSION  = "2022-06-28"
NOTION_BASE     = "https://api.notion.com/v1"


# ══════════════════════════════════════════════════════════════════════════
# 1. DOCUMENT TOKENISATION
# ══════════════════════════════════════════════════════════════════════════

def tokenise_document(text: str) -> dict:
    enc       = tiktoken.encoding_for_model(TOKENIZER_MODEL)
    token_ids = enc.encode(text)
    return {
        "token_count": len(token_ids),
        "tokens":      [enc.decode([t]) for t in token_ids],
    }

def print_tokenisation_info(text: str) -> None:
    r = tokenise_document(text)
    print("\n" + "─" * 50)
    print(f"Document tokenisation  ({TOKENIZER_MODEL} encoding)")
    print(f"   Characters : {len(text)}")
    print(f"   Tokens     : {r['token_count']}")
    print(f"   First 10   : {r['tokens'][:10]}")
    print("─" * 50 + "\n")


# ══════════════════════════════════════════════════════════════════════════
# 2. NOTION HELPERS  (internal — not exposed as tools)
# ══════════════════════════════════════════════════════════════════════════

def _headers() -> dict:
    return {
        "Authorization":  f"Bearer {NOTION_API_KEY}",
        "Content-Type":   "application/json",
        "Notion-Version": NOTION_VERSION,
    }

def _to_blocks(text: str) -> list:
    """Convert plain text into Notion paragraph blocks (max 2000 chars each)."""
    blocks = []
    for line in text.split("\n"):
        for chunk in [line[i:i+2000] for i in range(0, max(len(line), 1), 2000)]:
            blocks.append({
                "object": "block",
                "type": "paragraph",
                "paragraph": {
                    "rich_text": [{"type": "text", "text": {"content": chunk}}]
                },
            })
    return blocks

def _blocks_to_text(blocks: list) -> str:
    """Extract plain text from Notion block objects."""
    lines = []
    for block in blocks:
        btype     = block.get("type", "")
        rich_text = block.get(btype, {}).get("rich_text", [])
        line      = "".join(rt.get("text", {}).get("content", "") for rt in rich_text)
        if line:
            lines.append(line)
    return "\n".join(lines)

def _get_page_title(page_data: dict) -> str:
    """Pull the title string out of a Notion page response."""
    props = page_data.get("properties", {})
    for key in ("title", "Name"):
        tlist = props.get(key, {}).get("title", [])
        if tlist:
            return tlist[0].get("text", {}).get("content", "Untitled")
    return "Untitled"


# ══════════════════════════════════════════════════════════════════════════
# 3. TOOLS
# ══════════════════════════════════════════════════════════════════════════

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a safe math expression and return the numeric result.
    Supports: +  -  *  /  **  sqrt  sin  cos  tan  log  pi  e
    Example:  '2 ** 10 + sqrt(144)'
    """
    try:
        ns = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
        ns["__builtins__"] = {}
        return f"Result: {eval(expression, ns)}"   # noqa: S307
    except Exception as exc:
        return f"Error: {exc}"


@tool
def notion_create_page(title: str, content: str = "") -> str:
    """
    Create a new Notion page under the configured parent page.
    Use this when the user wants to create or save a new document.

    Args:
        title   : the title of the new page
        content : optional body text to write into the page

    Returns a summary with the new page_id and URL.
    """
    body = {
        "parent": {"page_id": NOTION_PARENT_PAGE_ID},
        "properties": {
            "title": {"title": [{"text": {"content": title}}]}
        },
        "children": _to_blocks(content) if content else [],
    }
    resp = requests.post(f"{NOTION_BASE}/pages", headers=_headers(), json=body)
    if resp.status_code != 200:
        return f"Failed to create page ({resp.status_code}): {resp.text}"
    data = resp.json()
    return (
        f"Page created!\n"
        f"  page_id : {data['id']}\n"
        f"  title   : {title}\n"
        f"  url     : {data.get('url', 'N/A')}\n"
        f"  content : {len(content)} characters written."
    )


@tool
def notion_read_page(page_id: str) -> str:
    """
    Read and return the full text content of a Notion page.
    Use this when the user wants to view or retrieve a page.

    Args:
        page_id : the Notion page ID (32-char hex, found in the page URL)
    """
    # Fetch title
    pr = requests.get(f"{NOTION_BASE}/pages/{page_id}", headers=_headers())
    if pr.status_code != 200:
        return f"Failed to fetch page ({pr.status_code}): {pr.text}"
    page_title = _get_page_title(pr.json())

    # Fetch blocks
    br = requests.get(
        f"{NOTION_BASE}/blocks/{page_id}/children",
        headers=_headers(),
        params={"page_size": 100},
    )
    if br.status_code != 200:
        return f"Failed to fetch content ({br.status_code}): {br.text}"

    text = _blocks_to_text(br.json().get("results", []))
    if not text.strip():
        return f"Page '{page_title}' exists but has no text content."

    return f"Page: '{page_title}'\nID: {page_id}\n{'─'*40}\n{text}"


@tool
def notion_append_to_page(page_id: str, content: str) -> str:
    """
    Append new text to the bottom of an existing Notion page.
    Use this when the user wants to add content to an existing page.

    Args:
        page_id : the Notion page ID
        content : text to append
    """
    resp = requests.patch(
        f"{NOTION_BASE}/blocks/{page_id}/children",
        headers=_headers(),
        json={"children": _to_blocks(content)},
    )
    if resp.status_code != 200:
        return f"Failed to append ({resp.status_code}): {resp.text}"
    return f"Appended {len(content)} characters to page '{page_id}' successfully."


@tool
def notion_search_pages(query: str) -> str:
    """
    Search for Notion pages by title keyword.
    Use this when the user wants to find an existing page by name.

    Args:
        query : keyword to search for in page titles
    """
    resp = requests.post(
        f"{NOTION_BASE}/search",
        headers=_headers(),
        json={"query": query, "filter": {"value": "page", "property": "object"}},
    )
    if resp.status_code != 200:
        return f"Search failed ({resp.status_code}): {resp.text}"

    results = resp.json().get("results", [])
    if not results:
        return f"No pages found matching '{query}'."

    lines = [f"Found {len(results)} page(s) matching '{query}':"]
    for r in results[:5]:
        title = _get_page_title(r)
        lines.append(f"  - {title}  |  id: {r['id']}  |  url: {r.get('url','')}")
    return "\n".join(lines)


# ══════════════════════════════════════════════════════════════════════════
# 4. LLM + TOOLS SETUP
# ══════════════════════════════════════════════════════════════════════════

TOOLS = [calculator, notion_create_page, notion_read_page, notion_append_to_page, notion_search_pages]


def build_llm():
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0,
        api_key=GROQ_API_KEY,
    )
    return llm.bind_tools(TOOLS)


# ══════════════════════════════════════════════════════════════════════════
# 5. LANGGRAPH AGENT
# ══════════════════════════════════════════════════════════════════════════

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


def build_agent_graph(llm_with_tools):
    tool_node = ToolNode(TOOLS)

    def agent_node(state: AgentState):
        return {"messages": [llm_with_tools.invoke(state["messages"])]}

    def should_continue(state: AgentState) -> str:
        last = state["messages"][-1]
        return "tools" if (hasattr(last, "tool_calls") and last.tool_calls) else END

    graph = StateGraph(AgentState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", tool_node)
    graph.set_entry_point("agent")
    graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
    graph.add_edge("tools", "agent")
    return graph.compile()


# ══════════════════════════════════════════════════════════════════════════
# 6. MAIN WHILE LOOP
# ══════════════════════════════════════════════════════════════════════════

SYSTEM_PROMPT = """You are a helpful assistant with access to these tools:

1. calculator           - evaluate any math expression
2. notion_create_page   - create a new Notion page with a title and content
3. notion_read_page     - read the content of a Notion page by page_id
4. notion_append_to_page- append text to an existing Notion page
5. notion_search_pages  - search for Notion pages by keyword

Always use the right tool for the task. When creating a page, use a descriptive title."""


def run_agent():
    llm = build_llm()
    app = build_agent_graph(llm)
    history = [SystemMessage(content=SYSTEM_PROMPT)]

    print("╔══════════════════════════════════════════════════╗")
    print("║  LangGraph + Groq + Notion  |  'exit' to quit   ║")
    print("╚══════════════════════════════════════════════════╝\n")
    print("Try:")
    print("  'create a notion page about machine learning'")
    print("  'search for pages about AI'")
    print("  'read page <page_id>'")
    print("  'what is 2 ** 16'\n")

    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("exit", "quit", "q"):
            print("Bye!")
            break

        print_tokenisation_info(user_input)
        history.append(HumanMessage(content=user_input))

        steps = 0
        state = {"messages": history}

        for event in app.stream(state, stream_mode="values"):
            steps += 1
            last_msg = event["messages"][-1]

            if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
                for tc in last_msg.tool_calls:
                    print(f"  Tool call   -> {tc['name']}({tc['args']})")

            if last_msg.type == "tool":
                print(f"  Tool result -> {last_msg.content}\n")

            if steps >= MAX_AGENT_STEPS:
                print("  Max steps reached.")
                break

        print(f"\nAssistant: {event['messages'][-1].content}\n")
        history = event["messages"]


if __name__ == "__main__":
    run_agent()

╔══════════════════════════════════════════════════╗
║  LangGraph + Groq + Notion  |  'exit' to quit   ║
╚══════════════════════════════════════════════════╝

Try:
  'create a notion page about machine learning'
  'search for pages about AI'
  'read page <page_id>'
  'what is 2 ** 16'



You:  write notion page about machine learning



──────────────────────────────────────────────────
Document tokenisation  (gpt-4o encoding)
   Characters : 40
   Tokens     : 6
   First 10   : ['write', ' notion', ' page', ' about', ' machine', ' learning']
──────────────────────────────────────────────────

  Tool call   -> notion_create_page({'content': 'Machine learning is a subset of artificial intelligence that involves the use of algorithms and statistical models to enable machines to perform a specific task without using explicit instructions. It is a key driver of modern technologies such as image and speech recognition, natural language processing, and predictive analytics.', 'title': 'Introduction to Machine Learning'})
  Tool result -> Page created!
  page_id : 35a42318-ab7e-814f-bf50-cde9485d62cc
  title   : Introduction to Machine Learning
  url     : https://www.notion.so/Introduction-to-Machine-Learning-35a42318ab7e814fbf50cde9485d62cc
  content : 332 characters written.

  Tool call   -> notion_append_to_page({'cont

You:  write notion page about gen-ai



──────────────────────────────────────────────────
Document tokenisation  (gpt-4o encoding)
   Characters : 30
   Tokens     : 6
   First 10   : ['write', ' notion', ' page', ' about', ' gen', '-ai']
──────────────────────────────────────────────────

  Tool call   -> notion_create_page({'content': 'Generative AI refers to a type of artificial intelligence that is capable of generating new, original content, such as images, videos, music, and text. This is achieved through the use of complex algorithms and neural networks that are trained on large datasets of existing content.', 'title': 'Introduction to Generative AI'})
  Tool result -> Page created!
  page_id : 35a42318-ab7e-815b-9fa9-ec0a62b4b147
  title   : Introduction to Generative AI
  url     : https://www.notion.so/Introduction-to-Generative-AI-35a42318ab7e815b9fa9ec0a62b4b147
  content : 282 characters written.

  Tool call   -> notion_append_to_page({'content': 'Some common applications of generative AI include art and desi